# CPU analysis — visualization-ready

Visualization-ready analysis export. This notebook normalizes all benchmark sampler outputs into consistent metric names and exports tables for plotting.

In [10]:

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 240)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

DATASETS = ["mmlu", "hellaswag", "gsm8k", "dolly"]
SCORED_DATASETS = ["mmlu", "hellaswag", "gsm8k"]

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
print("ROOT:", ROOT)

PLATFORM = "cpu"
RESULTS_DIR = ROOT / "results" / "cpu"
PROCESSED_DIR = RESULTS_DIR / "processed" / "cpu"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("RESULTS_DIR:", RESULTS_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)


ROOT: /home/stephane/resource-efficiency-benchmark
RESULTS_DIR: /home/stephane/resource-efficiency-benchmark/results/cpu
PROCESSED_DIR: /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu


## Discover raw benchmark runs

In [11]:

def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()
    except Exception as exc:
        warnings.warn(f"Could not read {path}: {exc}")
        return pd.DataFrame()


def looks_like_run_dir(path: Path) -> bool:
    return path.is_dir() and any((path / f"{d}.csv").exists() or (path / f"{d}_samples.csv").exists() for d in DATASETS)


def discover_runs(results_dir: Path) -> pd.DataFrame:
    rows = []
    if not results_dir.exists():
        return pd.DataFrame(columns=["platform", "model_dir", "run_id", "run_dir"])
    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir() or model_dir.name == "processed":
            continue
        for run_dir in sorted(model_dir.iterdir()):
            if looks_like_run_dir(run_dir):
                rows.append({
                    "platform": PLATFORM,
                    "model_dir": model_dir.name,
                    "run_id": run_dir.name,
                    "run_dir": str(run_dir),
                })
    return pd.DataFrame(rows)


def load_raw_tables(runs_df: pd.DataFrame):
    summary_parts, sample_parts = [], []
    for _, run in runs_df.iterrows():
        run_dir = Path(run["run_dir"])
        for dataset in DATASETS:
            summary_path = run_dir / f"{dataset}.csv"
            samples_path = run_dir / f"{dataset}_samples.csv"
            sdf = read_csv_if_exists(summary_path)
            if not sdf.empty:
                sdf = sdf.copy()
                sdf["platform"] = PLATFORM
                sdf["dataset"] = dataset
                sdf["run_id"] = run["run_id"]
                sdf["run_dir"] = str(run_dir)
                sdf["model_dir"] = run["model_dir"]
                sdf["source_file"] = str(summary_path)
                summary_parts.append(sdf)
            sam = read_csv_if_exists(samples_path)
            if not sam.empty:
                sam = sam.copy()
                sam["platform"] = PLATFORM
                sam["dataset"] = dataset
                sam["run_id"] = run["run_id"]
                sam["run_dir"] = str(run_dir)
                sam["model_dir"] = run["model_dir"]
                sam["source_file"] = str(samples_path)
                sample_parts.append(sam)
    return (
        pd.concat(summary_parts, ignore_index=True) if summary_parts else pd.DataFrame(),
        pd.concat(sample_parts, ignore_index=True) if sample_parts else pd.DataFrame(),
    )

runs_df = discover_runs(RESULTS_DIR)
print("Discovered runs:", len(runs_df))
display(runs_df.head(30))

Discovered runs: 11


,platform,model_dir,run_id,run_dir
0,cpu,Llama-3.2-1B-Instruct-Q8_0,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...
1,cpu,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...
2,cpu,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-05-01_07-29-58,/home/stephane/resource-efficiency-benchmark/r...
3,cpu,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-05-01_08-47-34,/home/stephane/resource-efficiency-benchmark/r...
4,cpu,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...
5,cpu,Phi-4-mini-instruct-Q4_K_M,2026-04-30_11-07-28,/home/stephane/resource-efficiency-benchmark/r...
6,cpu,Phi-4-mini-instruct-Q4_K_M,2026-04-30_21-54-52,/home/stephane/resource-efficiency-benchmark/r...
7,cpu,Phi-4-mini-instruct-Q4_K_M,2026-05-01_07-29-58,/home/stephane/resource-efficiency-benchmark/r...
8,cpu,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...
9,cpu,gemma-3-1b-it-Q8_0,2026-05-01_09-38-06,/home/stephane/resource-efficiency-benchmark/r...


## Load raw summary and sample CSVs

In [12]:
raw_summary, raw_samples = load_raw_tables(runs_df)
print("raw_summary", raw_summary.shape)
print("raw_samples", raw_samples.shape)
display(raw_summary.head())
display(raw_samples.head())

raw_summary (29, 50)
raw_samples (22700, 32)


,Model name,Model path,Served model id,All served model ids,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak,platform,dataset,run_id,run_dir,model_dir,source_file,Dolly Served model id,Dolly All served model ids,Dolly TTFT ms mean,Dolly TTFT ms sd,Dolly End-to-end latency ms mean,Dolly End-to-end latency ms sd,Dolly Tokens/sec mean,Dolly Tokens/sec sd,Dolly CPU percent avg mean,Dolly CPU percent avg sd,Dolly Temp C avg mean,Dolly Temp C avg sd,Dolly Sequential sustained tokens/sec mean,Dolly Sequential sustained tokens/sec sd,Dolly Sequential sustained samples/sec mean,Dolly Sequential sustained samples/sec sd,Dolly CPU percent peak,Dolly RAM used MB peak,Dolly Process RAM MB peak,Dolly Temp C peak
0,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf | Llama-3.2-1B...,0.38,0.0,1764.2507,50.5659,1820.8756,51.0658,18.1393,0.1496,14.5760,0.5265,87.5627,0.9136,0.5437,0.0143,0.5383,0.0142,50.8,11004.54,2020.26,100.0,cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf | Llama-3.2-1B...,0.29,0.0,2407.5919,22.4174,2463.7357,22.4443,18.0173,0.1680,14.1960,0.3511,88.7103,0.5547,0.3997,0.0035,0.3997,0.0035,30.2,5578.20,2251.58,100.0,cpu,hellaswag,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf | Llama-3.2-1B...,0.14,0.0,933.7012,23.2409,3220.7195,32.3383,12.4660,0.1428,15.9904,0.3070,79.5491,0.9342,8.5791,0.0847,0.3068,0.0030,30.4,5254.59,1906.89,100.0,cpu,gsm8k,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cpu,dolly,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf | Llama-3.2-1B...,1694.4607,260.0751,8857.5051,633.2956,13.0534,0.3709,16.8698,0.7031,73.8459,3.2416,10.6181,0.3766,0.1129,0.0083,44.8,5987.68,2527.98,100.0
4,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf | Min...,0.62,0.0,6268.5256,72.7269,6390.6456,73.8215,8.2628,0.1008,12.5081,0.4001,80.1767,0.6949,0.1555,0.0018,0.1555,0.0018,33.1,5643.69,4399.39,100.0,cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Ministral-3-3B-Instruct-2512-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,model_name,served_model_id,model_path,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,system_metrics_trace_json,platform,dataset,run_id,run_dir,model_dir,source_file
0,Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,mmlu,1,1,A disk of radius 0.1 m initially at rest under...,A disk of radius 0.1 m initially at rest under...,0.4 m || 0.8 m || 1.2 m || 1.6 m,NaN,NaN,3.0,C,False,C,1,1194.3270,1246.8526,19.0383,13.5667,20.3,5020.76,1570.88,88.6667,94.0,"[{""cpu_percent"":2.4,""ram_used_mb"":5013.31,""ram...",cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...
1,Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,mmlu,1,2,"Cultural homogenization.\n\nA. 1,3,4\nB. 1,2,3...",Cultural homogenization.,"1,3,4 || 1,2,3 || 1,2,3,4 || 2,3,4",NaN,NaN,1.0,C,False,C,1,782.7442,831.8889,20.3481,14.4667,20.7,5024.71,1574.49,94.0000,94.0,"[{""cpu_percent"":4.7,""ram_used_mb"":5020.76,""ram...",cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...
2,Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,mmlu,1,3,Monte Alban is located in the ________ and was...,Monte Alban is located in the ________ and was...,Valley of Mexico; Aztec. || Valley of Oaxaca; ...,NaN,NaN,1.0,D,False,D,1,905.1569,955.9261,19.6970,15.7333,21.7,5025.82,1576.82,95.3333,96.0,"[{""cpu_percent"":7.0,""ram_used_mb"":5025.39,""ram...",cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...
3,Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,mmlu,1,4,During a clinical study examining the effects ...,During a clinical study examining the effects ...,Pulse pressure || Stroke volume || Systolic bl...,NaN,NaN,3.0,C,False,C,1,1452.1402,1502.7018,19.7779,14.7333,18.4,5030.75,1579.41,98.6667,100.0,"[{""cpu_percent"":7.5,""ram_used_mb"":5025.82,""ram...",cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...
4,Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,mmlu,1,5,A 17-year-old girl is brought to the physician...,A 17-year-old girl is brought to the physician...,Hyperthyroidism || Hypogonadotropic hypogonadi...,NaN,NaN,1.0,B,True,B,1,2387.0870,2439.8306,18.9596,14.6000,19.4,5030.75,1585.43,97.3333,100.0,"[{""cpu_percent"":6.0,""ram_used_mb"":5030.75,""ram...",cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...


## Normalize schema and derive visualization metrics

In [13]:

def collapse_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Merge duplicate column names by taking the first non-null value row-wise."""
    if df.empty:
        return df.copy()
    out = pd.DataFrame(index=df.index)
    for col in dict.fromkeys(df.columns):
        block = df.loc[:, df.columns == col]
        if block.shape[1] == 1:
            out[col] = block.iloc[:, 0]
        else:
            out[col] = block.bfill(axis=1).iloc[:, 0]
    return out.reset_index(drop=True)


def first_existing(df: pd.DataFrame, candidates, default=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return pd.Series(default, index=df.index)


def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def to_bool_or_nan(s):
    if s is None:
        return np.nan
    if not isinstance(s, pd.Series):
        return s
    def conv(x):
        if pd.isna(x):
            return np.nan
        if isinstance(x, bool):
            return x
        text = str(x).strip().lower()
        if text in {"true", "1", "yes", "y"}:
            return True
        if text in {"false", "0", "no", "n"}:
            return False
        return np.nan
    return s.map(conv)


def safe_div(num, den):
    num = to_num(num)
    den = to_num(den)
    return np.where((den.notna()) & (den != 0), num / den, np.nan)


def add_common_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = collapse_duplicate_columns(df)
    if df.empty:
        return df
    if "platform" not in df.columns:
        df["platform"] = PLATFORM
    if "dataset" in df.columns:
        df["dataset"] = df["dataset"].astype(str).str.lower()
    if "model" not in df.columns:
        df["model"] = first_existing(df, ["model_name", "Model name", "model_alias", "Model alias", "model_dir", "model_alias_dir", "Repo id", "repo_id"])
    df["model"] = df["model"].astype(str)
    df["model_label"] = df["platform"].astype(str) + " | " + df["model"].astype(str)
    if "run_id" in df.columns:
        df["run_label"] = df["platform"].astype(str) + " | " + df["model"].astype(str) + " | " + df["run_id"].astype(str)
    return df

def normalize_summary(raw: pd.DataFrame) -> pd.DataFrame:
    raw = add_common_labels(raw)
    if raw.empty:
        return raw
    out = pd.DataFrame(index=raw.index)
    for col in ["platform", "dataset", "run_id", "run_dir", "model", "model_label", "model_dir", "source_file"]:
        if col in raw.columns:
            out[col] = raw[col]

    # llama.cpp summaries use human-readable names; Dolly may be prefixed with "Dolly ".
    out["accuracy"] = to_num(first_existing(raw, ["Accuracy mean", "Dolly Accuracy mean"]))
    out["accuracy_sd"] = to_num(first_existing(raw, ["Accuracy sd", "Dolly Accuracy sd"]))
    out["ttft_ms"] = to_num(first_existing(raw, ["TTFT ms mean", "Dolly TTFT ms mean"]))
    out["ttft_ms_sd"] = to_num(first_existing(raw, ["TTFT ms sd", "Dolly TTFT ms sd"]))
    out["latency_ms"] = to_num(first_existing(raw, ["End-to-end latency ms mean", "Dolly End-to-end latency ms mean"]))
    out["latency_ms_sd"] = to_num(first_existing(raw, ["End-to-end latency ms sd", "Dolly End-to-end latency ms sd"]))
    out["tokens_per_sec"] = to_num(first_existing(raw, ["Tokens/sec mean", "Dolly Tokens/sec mean"]))
    out["tokens_per_sec_sd"] = to_num(first_existing(raw, ["Tokens/sec sd", "Dolly Tokens/sec sd"]))
    out["sustained_tokens_per_sec"] = to_num(first_existing(raw, ["Sequential sustained tokens/sec mean", "Dolly Sequential sustained tokens/sec mean"]))
    out["sustained_tokens_per_sec_sd"] = to_num(first_existing(raw, ["Sequential sustained tokens/sec sd", "Dolly Sequential sustained tokens/sec sd"]))
    out["samples_per_sec"] = to_num(first_existing(raw, ["Sequential sustained samples/sec mean", "Dolly Sequential sustained samples/sec mean"]))
    out["samples_per_sec_sd"] = to_num(first_existing(raw, ["Sequential sustained samples/sec sd", "Dolly Sequential sustained samples/sec sd"]))
    out["cpu_percent_avg"] = to_num(first_existing(raw, ["CPU percent avg mean", "Dolly CPU percent avg mean"]))
    out["cpu_percent_avg_sd"] = to_num(first_existing(raw, ["CPU percent avg sd", "Dolly CPU percent avg sd"]))
    out["cpu_percent_peak"] = to_num(first_existing(raw, ["CPU percent peak", "Dolly CPU percent peak"]))
    out["ram_used_mb_peak"] = to_num(first_existing(raw, ["RAM used MB peak", "Dolly RAM used MB peak"]))
    out["process_ram_mb_peak"] = to_num(first_existing(raw, ["Process RAM MB peak", "Dolly Process RAM MB peak"]))
    out["temperature_c_avg"] = to_num(first_existing(raw, ["Temp C avg mean", "Dolly Temp C avg mean"]))
    out["temperature_c_avg_sd"] = to_num(first_existing(raw, ["Temp C avg sd", "Dolly Temp C avg sd"]))
    out["temperature_c_peak"] = to_num(first_existing(raw, ["Temp C peak", "Dolly Temp C peak"]))

    out["has_accuracy"] = out["dataset"].isin(SCORED_DATASETS) & out["accuracy"].notna()
    out["accuracy_per_second"] = safe_div(out["accuracy"], out["latency_ms"] / 1000.0)
    out["accuracy_per_1k_ms"] = safe_div(out["accuracy"], out["latency_ms"] / 1000.0)  # same unit as per-second utility
    return out


def normalize_samples(raw: pd.DataFrame) -> pd.DataFrame:
    raw = add_common_labels(raw)
    if raw.empty:
        return raw
    out = pd.DataFrame(index=raw.index)
    keep_cols = [
        "platform", "dataset", "run_id", "run_dir", "model", "model_label", "model_dir", "source_file",
        "served_model_id", "model_path", "sample_idx", "prompt", "question", "choices", "instruction", "context", "gold_answer",
        "parsed_prediction", "raw_output", "system_metrics_trace_json",
    ]
    for col in keep_cols:
        if col in raw.columns:
            out[col] = raw[col]
    out["correct"] = to_bool_or_nan(first_existing(raw, ["correct"]))
    out["accuracy"] = out["correct"].astype(float)
    out.loc[~out["dataset"].isin(SCORED_DATASETS), "accuracy"] = np.nan
    out["output_tokens"] = to_num(first_existing(raw, ["output_tokens"]))
    out["ttft_ms"] = to_num(first_existing(raw, ["ttft_ms"]))
    out["latency_ms"] = to_num(first_existing(raw, ["end_to_end_latency_ms", "latency_ms"]))
    out["tokens_per_sec"] = to_num(first_existing(raw, ["tokens_per_sec"]))
    out["cpu_percent_avg"] = to_num(first_existing(raw, ["cpu_percent_avg"]))
    out["cpu_percent_peak"] = to_num(first_existing(raw, ["cpu_percent_peak"]))
    out["ram_used_mb_peak"] = to_num(first_existing(raw, ["ram_used_mb_peak"]))
    out["process_ram_mb_peak"] = to_num(first_existing(raw, ["process_ram_mb_peak"]))
    out["temperature_c_avg"] = to_num(first_existing(raw, ["temp_c_avg", "temperature_c_avg"]))
    out["temperature_c_peak"] = to_num(first_existing(raw, ["temp_c_peak", "temperature_c_peak"]))
    out["accuracy_per_second"] = safe_div(out["accuracy"], out["latency_ms"] / 1000.0)
    return out


In [14]:

def expand_system_traces(samples_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if samples_df.empty or "system_metrics_trace_json" not in samples_df.columns:
        return pd.DataFrame()
    meta_cols = ["platform", "dataset", "model", "model_label", "run_id", "sample_idx"]
    for _, row in samples_df.iterrows():
        raw = row.get("system_metrics_trace_json")
        if pd.isna(raw) or raw in ("", None):
            continue
        try:
            points = json.loads(raw)
        except Exception:
            continue
        if not isinstance(points, list):
            continue
        for trace_idx, point in enumerate(points):
            if not isinstance(point, dict):
                continue
            rec = {c: row.get(c) for c in meta_cols if c in samples_df.columns}
            rec.update({
                "trace_idx": trace_idx,
                "t_s": point.get("t_s", np.nan),
                "phase": point.get("phase", np.nan),
                "cpu_percent": point.get("cpu_percent"),
                "ram_used_mb": point.get("ram_used_mb"),
                "ram_percent": point.get("ram_percent"),
                "process_ram_mb": point.get("process_ram_mb"),
                "temperature_c": point.get("temperature_c"),
            })
            rows.append(rec)
    traces = pd.DataFrame(rows)
    for col in ["trace_idx", "t_s", "cpu_percent", "ram_used_mb", "ram_percent", "process_ram_mb", "temperature_c"]:
        if col in traces.columns:
            traces[col] = to_num(traces[col])
    return traces


def summarize_for_visualization(summary: pd.DataFrame, samples: pd.DataFrame, traces: pd.DataFrame):
    group_cols = ["platform", "dataset", "model", "model_label"]
    sample_aggs = {
        "sample_count": ("sample_idx", "count"),
        "accuracy": ("accuracy", "mean"),
        "latency_ms_mean": ("latency_ms", "mean"),
        "latency_ms_median": ("latency_ms", "median"),
        "latency_ms_p95": ("latency_ms", lambda x: x.quantile(0.95)),
        "ttft_ms_mean": ("ttft_ms", "mean"),
        "tokens_per_sec_mean": ("tokens_per_sec", "mean"),
        "tokens_per_sec_median": ("tokens_per_sec", "median"),
        "output_tokens_mean": ("output_tokens", "mean"),
        "cpu_percent_avg_mean": ("cpu_percent_avg", "mean"),
        "cpu_percent_peak_max": ("cpu_percent_peak", "max"),
        "ram_used_mb_peak_max": ("ram_used_mb_peak", "max"),
        "process_ram_mb_peak_max": ("process_ram_mb_peak", "max"),
        "temperature_c_avg_mean": ("temperature_c_avg", "mean"),
        "temperature_c_peak_max": ("temperature_c_peak", "max"),
        "accuracy_per_second": ("accuracy_per_second", "mean"),
    }
    usable_aggs = {k: v for k, v in sample_aggs.items() if v[0] in samples.columns}
    model_dataset_summary = samples.groupby(group_cols, dropna=False).agg(**usable_aggs).reset_index() if not samples.empty and usable_aggs else pd.DataFrame()

    if not traces.empty:
        trace_summary = traces.groupby(group_cols, dropna=False).agg(
            trace_points=("trace_idx", "count"),
            trace_cpu_percent_mean=("cpu_percent", "mean"),
            trace_cpu_percent_peak=("cpu_percent", "max"),
            trace_ram_used_mb_mean=("ram_used_mb", "mean"),
            trace_ram_used_mb_peak=("ram_used_mb", "max"),
            trace_process_ram_mb_mean=("process_ram_mb", "mean"),
            trace_process_ram_mb_peak=("process_ram_mb", "max"),
            trace_temperature_c_mean=("temperature_c", "mean"),
            trace_temperature_c_peak=("temperature_c", "max"),
        ).reset_index()
    else:
        trace_summary = pd.DataFrame()

    ranking_cols = ["platform", "dataset", "model", "model_label", "accuracy", "latency_ms_mean", "tokens_per_sec_mean", "ram_used_mb_peak_max", "temperature_c_peak_max", "accuracy_per_second"]
    ranking = model_dataset_summary[[c for c in ranking_cols if c in model_dataset_summary.columns]].copy() if not model_dataset_summary.empty else pd.DataFrame()
    if not ranking.empty:
        ranking = ranking.sort_values(["dataset", "accuracy", "tokens_per_sec_mean", "latency_ms_mean"], ascending=[True, False, False, True])
    return model_dataset_summary, trace_summary, ranking


summary = normalize_summary(raw_summary)
samples = normalize_samples(raw_samples)
system_traces = expand_system_traces(samples)
model_dataset_summary, system_trace_summary, ranking_by_dataset = summarize_for_visualization(summary, samples, system_traces)

print("summary", summary.shape)
print("samples", samples.shape)
print("system_traces", system_traces.shape)
print("model_dataset_summary", model_dataset_summary.shape)
display(summary.head())
display(samples.head())
display(model_dataset_summary.head())


summary (29, 31)
samples (22700, 33)
system_traces (629740, 14)
model_dataset_summary (16, 20)


,platform,dataset,run_id,run_dir,model,model_label,model_dir,source_file,accuracy,accuracy_sd,ttft_ms,ttft_ms_sd,latency_ms,latency_ms_sd,tokens_per_sec,tokens_per_sec_sd,sustained_tokens_per_sec,sustained_tokens_per_sec_sd,samples_per_sec,samples_per_sec_sd,cpu_percent_avg,cpu_percent_avg_sd,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temperature_c_avg,temperature_c_avg_sd,temperature_c_peak,has_accuracy,accuracy_per_second,accuracy_per_1k_ms
0,cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,0.38,0.0,1764.2507,50.5659,1820.8756,51.0658,18.1393,0.1496,0.5437,0.0143,0.5383,0.0142,14.5760,0.5265,50.8,11004.54,2020.26,87.5627,0.9136,100.0,True,0.208691,0.208691
1,cpu,hellaswag,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,0.29,0.0,2407.5919,22.4174,2463.7357,22.4443,18.0173,0.1680,0.3997,0.0035,0.3997,0.0035,14.1960,0.3511,30.2,5578.20,2251.58,88.7103,0.5547,100.0,True,0.117707,0.117707
2,cpu,gsm8k,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,0.14,0.0,933.7012,23.2409,3220.7195,32.3383,12.4660,0.1428,8.5791,0.0847,0.3068,0.0030,15.9904,0.3070,30.4,5254.59,1906.89,79.5491,0.9342,100.0,True,0.043469,0.043469
3,cpu,dolly,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN
4,cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,cpu | Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...,0.62,0.0,6268.5256,72.7269,6390.6456,73.8215,8.2628,0.1008,0.1555,0.0018,0.1555,0.0018,12.5081,0.4001,33.1,5643.69,4399.39,80.1767,0.6949,100.0,True,0.097017,0.097017


,platform,dataset,run_id,run_dir,model,model_label,model_dir,source_file,served_model_id,model_path,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,raw_output,system_metrics_trace_json,correct,accuracy,output_tokens,ttft_ms,latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temperature_c_avg,temperature_c_peak,accuracy_per_second
0,cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,1,A disk of radius 0.1 m initially at rest under...,A disk of radius 0.1 m initially at rest under...,0.4 m || 0.8 m || 1.2 m || 1.6 m,NaN,NaN,3.0,C,C,"[{""cpu_percent"":2.4,""ram_used_mb"":5013.31,""ram...",False,0.0,1,1194.3270,1246.8526,19.0383,13.5667,20.3,5020.76,1570.88,88.6667,94.0,0.000000
1,cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,2,"Cultural homogenization.\n\nA. 1,3,4\nB. 1,2,3...",Cultural homogenization.,"1,3,4 || 1,2,3 || 1,2,3,4 || 2,3,4",NaN,NaN,1.0,C,C,"[{""cpu_percent"":4.7,""ram_used_mb"":5020.76,""ram...",False,0.0,1,782.7442,831.8889,20.3481,14.4667,20.7,5024.71,1574.49,94.0000,94.0,0.000000
2,cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,3,Monte Alban is located in the ________ and was...,Monte Alban is located in the ________ and was...,Valley of Mexico; Aztec. || Valley of Oaxaca; ...,NaN,NaN,1.0,D,D,"[{""cpu_percent"":7.0,""ram_used_mb"":5025.39,""ram...",False,0.0,1,905.1569,955.9261,19.6970,15.7333,21.7,5025.82,1576.82,95.3333,96.0,0.000000
3,cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,4,During a clinical study examining the effects ...,During a clinical study examining the effects ...,Pulse pressure || Stroke volume || Systolic bl...,NaN,NaN,3.0,C,C,"[{""cpu_percent"":7.5,""ram_used_mb"":5025.82,""ram...",False,0.0,1,1452.1402,1502.7018,19.7779,14.7333,18.4,5030.75,1579.41,98.6667,100.0,0.000000
4,cpu,mmlu,2026-04-29_12-47-45,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,Llama-3.2-1B-Instruct-Q8_0.gguf,/app/models/Llama-3.2-1B-Instruct-Q8_0.gguf,5,A 17-year-old girl is brought to the physician...,A 17-year-old girl is brought to the physician...,Hyperthyroidism || Hypogonadotropic hypogonadi...,NaN,NaN,1.0,B,B,"[{""cpu_percent"":6.0,""ram_used_mb"":5030.75,""ram...",True,1.0,1,2387.0870,2439.8306,18.9596,14.6000,19.4,5030.75,1585.43,97.3333,100.0,0.409865


,platform,dataset,model,model_label,sample_count,accuracy,latency_ms_mean,latency_ms_median,latency_ms_p95,ttft_ms_mean,tokens_per_sec_mean,tokens_per_sec_median,output_tokens_mean,cpu_percent_avg_mean,cpu_percent_peak_max,ram_used_mb_peak_max,process_ram_mb_peak_max,temperature_c_avg_mean,temperature_c_peak_max,accuracy_per_second
0,cpu,dolly,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,1000,NaN,8857.505135,7961.23910,17231.078930,1694.352000,13.053870,13.29700,94.441000,16.869818,44.8,5987.68,2527.98,73.845910,100.0,NaN
1,cpu,dolly,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,cpu | Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,1100,NaN,24448.798803,26964.85290,38984.253095,3947.927832,5.895737,5.89150,120.427273,76.473908,94.1,12296.67,6207.45,66.704244,100.0,NaN
2,cpu,dolly,Phi-4-mini-instruct-Q4_K_M.gguf,cpu | Phi-4-mini-instruct-Q4_K_M.gguf,1100,NaN,20228.520654,21296.29955,35221.956185,4042.694501,7.155577,7.29445,117.470000,80.235026,96.6,14641.50,8977.90,67.042987,100.0,NaN
3,cpu,dolly,gemma-3-1b-it-Q8_0.gguf,cpu | gemma-3-1b-it-Q8_0.gguf,1100,NaN,9668.709704,10569.65230,15116.554485,1175.895884,13.145334,12.82945,111.610000,65.360685,88.2,13267.50,2477.81,63.543301,100.0,NaN
4,cpu,gsm8k,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,1000,0.14,3220.719461,1871.68250,12489.398970,933.701231,12.466029,12.30165,27.960000,15.990431,30.4,5254.59,1906.89,79.549146,100.0,0.059743


## Metric catalog with units

In [15]:

METRIC_CATALOG = pd.DataFrame([
    {"metric": "accuracy", "unit": "fraction", "level": "sample/summary", "meaning": "Correctness rate; only MMLU/HellaSwag/GSM8K"},
    {"metric": "latency_ms", "unit": "ms", "level": "sample/summary", "meaning": "End-to-end latency per request"},
    {"metric": "ttft_ms", "unit": "ms", "level": "sample/summary", "meaning": "Time to first token"},
    {"metric": "tokens_per_sec", "unit": "tokens/s", "level": "sample/summary", "meaning": "Decode throughput after first token"},
    {"metric": "sustained_tokens_per_sec", "unit": "tokens/s", "level": "summary", "meaning": "Sequential sustained throughput over full measured dataset run"},
    {"metric": "samples_per_sec", "unit": "samples/s", "level": "summary", "meaning": "Sequential sustained sample throughput"},
    {"metric": "output_tokens", "unit": "tokens", "level": "sample", "meaning": "Whitespace-estimated generated tokens"},
    {"metric": "cpu_percent_avg", "unit": "%", "level": "sample/summary", "meaning": "Average CPU utilization sampled during generation"},
    {"metric": "cpu_percent_peak", "unit": "%", "level": "sample/summary", "meaning": "Peak CPU utilization sampled during generation"},
    {"metric": "ram_used_mb_peak", "unit": "MB", "level": "sample/summary", "meaning": "Peak system RAM used"},
    {"metric": "process_ram_mb_peak", "unit": "MB", "level": "sample/summary", "meaning": "Peak llama-server process RSS"},
    {"metric": "temperature_c_avg", "unit": "°C", "level": "sample/summary", "meaning": "Average temperature sampled during generation"},
    {"metric": "temperature_c_peak", "unit": "°C", "level": "sample/summary", "meaning": "Peak temperature sampled during generation"},
    {"metric": "accuracy_per_second", "unit": "accuracy/s", "level": "derived", "meaning": "Accuracy divided by latency seconds; only scored datasets"},
])

display(METRIC_CATALOG)


,metric,unit,level,meaning
0,accuracy,fraction,sample/summary,Correctness rate; only MMLU/HellaSwag/GSM8K
1,latency_ms,ms,sample/summary,End-to-end latency per request
2,ttft_ms,ms,sample/summary,Time to first token
3,tokens_per_sec,tokens/s,sample/summary,Decode throughput after first token
4,sustained_tokens_per_sec,tokens/s,summary,Sequential sustained throughput over full meas...
5,samples_per_sec,samples/s,summary,Sequential sustained sample throughput
6,output_tokens,tokens,sample,Whitespace-estimated generated tokens
7,cpu_percent_avg,%,sample/summary,Average CPU utilization sampled during generation
8,cpu_percent_peak,%,sample/summary,Peak CPU utilization sampled during generation
9,ram_used_mb_peak,MB,sample/summary,Peak system RAM used


## Quick validation views

In [16]:

print("Available datasets in samples:", sorted(samples["dataset"].dropna().unique()) if "dataset" in samples.columns and not samples.empty else [])
print("Scored datasets:", SCORED_DATASETS)

if not ranking_by_dataset.empty:
    display(ranking_by_dataset.head(30))

for dataset in DATASETS:
    if not model_dataset_summary.empty and dataset in set(model_dataset_summary["dataset"]):
        print(f"\n{dataset.upper()} model/dataset summary")
        display(model_dataset_summary[model_dataset_summary["dataset"] == dataset].head(20))


Available datasets in samples: ['dolly', 'gsm8k', 'hellaswag', 'mmlu']
Scored datasets: ['mmlu', 'hellaswag', 'gsm8k']


,platform,dataset,model,model_label,accuracy,latency_ms_mean,tokens_per_sec_mean,ram_used_mb_peak_max,temperature_c_peak_max,accuracy_per_second
3,cpu,dolly,gemma-3-1b-it-Q8_0.gguf,cpu | gemma-3-1b-it-Q8_0.gguf,NaN,9668.709704,13.145334,13267.50,100.0,NaN
0,cpu,dolly,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,NaN,8857.505135,13.053870,5987.68,100.0,NaN
2,cpu,dolly,Phi-4-mini-instruct-Q4_K_M.gguf,cpu | Phi-4-mini-instruct-Q4_K_M.gguf,NaN,20228.520654,7.155577,14641.50,100.0,NaN
1,cpu,dolly,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,cpu | Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,NaN,24448.798803,5.895737,12296.67,100.0,NaN
6,cpu,gsm8k,Phi-4-mini-instruct-Q4_K_M.gguf,cpu | Phi-4-mini-instruct-Q4_K_M.gguf,0.640000,18628.568073,7.440439,13777.97,100.0,0.045347
5,cpu,gsm8k,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,cpu | Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,0.524545,24322.157501,5.444112,12899.34,100.0,0.026876
7,cpu,gsm8k,gemma-3-1b-it-Q8_0.gguf,cpu | gemma-3-1b-it-Q8_0.gguf,0.472727,9805.159573,15.354863,13044.35,100.0,0.055422
4,cpu,gsm8k,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,0.140000,3220.719461,12.466029,5254.59,100.0,0.059743
10,cpu,hellaswag,Phi-4-mini-instruct-Q4_K_M.gguf,cpu | Phi-4-mini-instruct-Q4_K_M.gguf,0.510000,6391.013000,8.275144,13269.09,100.0,0.091800
9,cpu,hellaswag,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,cpu | Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,0.405000,7681.311507,8.210181,10717.30,100.0,0.065501



MMLU model/dataset summary


,platform,dataset,model,model_label,sample_count,accuracy,latency_ms_mean,latency_ms_median,latency_ms_p95,ttft_ms_mean,tokens_per_sec_mean,tokens_per_sec_median,output_tokens_mean,cpu_percent_avg_mean,cpu_percent_peak_max,ram_used_mb_peak_max,process_ram_mb_peak_max,temperature_c_avg_mean,temperature_c_peak_max,accuracy_per_second
12,cpu,mmlu,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,1000,0.38,1820.875626,1328.40780,4726.002830,1764.250700,18.139343,18.68920,1.010,14.575966,50.8,11004.54,2020.26,87.562667,100.0,0.288077
13,cpu,mmlu,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,cpu | Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,2000,0.63,5702.324979,4313.22985,13240.215325,5580.990516,8.347900,8.16275,1.000,34.581084,94.4,10818.85,4812.91,73.376500,100.0,0.156950
14,cpu,mmlu,Phi-4-mini-instruct-Q4_K_M.gguf,cpu | Phi-4-mini-instruct-Q4_K_M.gguf,3000,0.54,4396.022804,3304.30535,10293.651015,4079.376202,8.680161,8.12240,2.516,57.154097,96.0,12329.49,6685.62,80.207383,100.0,0.178530
15,cpu,mmlu,gemma-3-1b-it-Q8_0.gguf,cpu | gemma-3-1b-it-Q8_0.gguf,1100,0.31,1142.733211,879.29760,2511.655980,1062.649114,12.996032,13.61825,1.000,49.128789,85.1,11657.05,2011.58,77.875758,100.0,0.382470



HELLASWAG model/dataset summary


,platform,dataset,model,model_label,sample_count,accuracy,latency_ms_mean,latency_ms_median,latency_ms_p95,ttft_ms_mean,tokens_per_sec_mean,tokens_per_sec_median,output_tokens_mean,cpu_percent_avg_mean,cpu_percent_peak_max,ram_used_mb_peak_max,process_ram_mb_peak_max,temperature_c_avg_mean,temperature_c_peak_max,accuracy_per_second
8,cpu,hellaswag,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,1000,0.290,2463.735679,2556.19160,4266.856615,2407.591918,18.017257,18.51130,1.000,14.196032,30.2,5578.20,2251.58,88.710333,100.0,0.141180
9,cpu,hellaswag,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,cpu | Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,2000,0.405,7681.311507,8116.82585,12811.225740,7558.017489,8.210181,8.08100,1.000,34.707533,94.5,10717.30,4426.10,73.471166,100.0,0.065501
10,cpu,hellaswag,Phi-4-mini-instruct-Q4_K_M.gguf,cpu | Phi-4-mini-instruct-Q4_K_M.gguf,2000,0.510,6391.013000,5621.99635,10213.645380,6029.228286,8.275144,8.15555,2.976,56.838051,96.2,13269.09,7626.35,75.869583,100.0,0.091800
11,cpu,hellaswag,gemma-3-1b-it-Q8_0.gguf,cpu | gemma-3-1b-it-Q8_0.gguf,1100,0.290,1474.606602,1487.75145,2543.916130,1396.670013,13.124227,13.77145,1.000,49.891031,84.1,12305.72,2388.45,78.254847,100.0,0.225161



GSM8K model/dataset summary


,platform,dataset,model,model_label,sample_count,accuracy,latency_ms_mean,latency_ms_median,latency_ms_p95,ttft_ms_mean,tokens_per_sec_mean,tokens_per_sec_median,output_tokens_mean,cpu_percent_avg_mean,cpu_percent_peak_max,ram_used_mb_peak_max,process_ram_mb_peak_max,temperature_c_avg_mean,temperature_c_peak_max,accuracy_per_second
4,cpu,gsm8k,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,1000,0.140000,3220.719461,1871.68250,12489.398970,933.701231,12.466029,12.30165,27.960000,15.990431,30.4,5254.59,1906.89,79.549146,100.0,0.059743
5,cpu,gsm8k,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,cpu | Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,1100,0.524545,24322.157501,26362.85030,31162.916415,2480.274566,5.444112,5.27670,118.628182,76.921381,98.8,12899.34,4434.47,65.367204,100.0,0.026876
6,cpu,gsm8k,Phi-4-mini-instruct-Q4_K_M.gguf,cpu | Phi-4-mini-instruct-Q4_K_M.gguf,2000,0.640000,18628.568073,18198.74120,30265.228460,2412.321572,7.440439,7.33920,117.237000,81.308880,94.4,13777.97,8123.45,67.369504,100.0,0.045347
7,cpu,gsm8k,gemma-3-1b-it-Q8_0.gguf,cpu | gemma-3-1b-it-Q8_0.gguf,1100,0.472727,9805.159573,10421.01275,11834.734070,634.713290,15.354863,15.30325,139.458182,69.445579,82.2,13044.35,2214.18,67.811124,100.0,0.055422



DOLLY model/dataset summary


,platform,dataset,model,model_label,sample_count,accuracy,latency_ms_mean,latency_ms_median,latency_ms_p95,ttft_ms_mean,tokens_per_sec_mean,tokens_per_sec_median,output_tokens_mean,cpu_percent_avg_mean,cpu_percent_peak_max,ram_used_mb_peak_max,process_ram_mb_peak_max,temperature_c_avg_mean,temperature_c_peak_max,accuracy_per_second
0,cpu,dolly,Llama-3.2-1B-Instruct-Q8_0.gguf,cpu | Llama-3.2-1B-Instruct-Q8_0.gguf,1000,NaN,8857.505135,7961.23910,17231.078930,1694.352000,13.053870,13.29700,94.441000,16.869818,44.8,5987.68,2527.98,73.845910,100.0,NaN
1,cpu,dolly,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,cpu | Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,1100,NaN,24448.798803,26964.85290,38984.253095,3947.927832,5.895737,5.89150,120.427273,76.473908,94.1,12296.67,6207.45,66.704244,100.0,NaN
2,cpu,dolly,Phi-4-mini-instruct-Q4_K_M.gguf,cpu | Phi-4-mini-instruct-Q4_K_M.gguf,1100,NaN,20228.520654,21296.29955,35221.956185,4042.694501,7.155577,7.29445,117.470000,80.235026,96.6,14641.50,8977.90,67.042987,100.0,NaN
3,cpu,dolly,gemma-3-1b-it-Q8_0.gguf,cpu | gemma-3-1b-it-Q8_0.gguf,1100,NaN,9668.709704,10569.65230,15116.554485,1175.895884,13.145334,12.82945,111.610000,65.360685,88.2,13267.50,2477.81,63.543301,100.0,NaN


## Export visualization-ready tables

In [17]:

EXPORTS = {
    "runs.csv": runs_df,
    "raw_summary.csv": raw_summary,
    "raw_samples.csv": raw_samples,
    "summary.csv": summary,
    "samples.csv": samples,
    "system_traces.csv": system_traces,
    "system_trace_summary.csv": system_trace_summary,
    "model_dataset_summary.csv": model_dataset_summary,
    "ranking_by_dataset.csv": ranking_by_dataset,
    "metric_catalog.csv": METRIC_CATALOG,
}

for filename, df in EXPORTS.items():
    path = PROCESSED_DIR / filename
    df.to_csv(path, index=False)
    print("wrote", path)

try:
    for filename, df in EXPORTS.items():
        df.to_parquet(PROCESSED_DIR / filename.replace(".csv", ".parquet"), index=False)
    print("Parquet export complete")
except Exception as exc:
    print("Skipped parquet export:", exc)


wrote /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu/runs.csv
wrote /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu/raw_summary.csv
wrote /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu/raw_samples.csv
wrote /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu/summary.csv
wrote /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu/samples.csv
wrote /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu/system_traces.csv
wrote /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu/system_trace_summary.csv
wrote /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu/model_dataset_summary.csv
wrote /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu/ranking_by_dataset.csv
wrote /home/stephane/resource-efficiency-benchmark/results/cpu/processed/cpu/metric_catalog.csv
Skipped parquet export: ("Expected bytes, got a 'float

## VS Code export helper

Run the next cell to export this notebook to HTML from VS Code/Jupyter.

In [18]:
import subprocess, sys
notebook_name = Path(globals().get("__vsc_ipynb_file__", "01_results_analysis_cpu_visualization_ready.ipynb"))
try:
    subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "html", str(notebook_name)], check=True)
except Exception as exc:
    print("HTML export skipped. In VS Code, use: Jupyter: Export to HTML")
    print(exc)

HTML export skipped. In VS Code, use: Jupyter: Export to HTML
Command '['/home/stephane/resource-efficiency-benchmark/.venv/bin/python', '-m', 'jupyter', 'nbconvert', '--to', 'html', '/home/stephane/resource-efficiency-benchmark/notebooks/01_results_analysis_cpu_visualization_ready.ipynb']' returned non-zero exit status 1.


usage: jupyter.py [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
                  [--paths] [--json] [--debug]
                  [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: kernel kernelspec migrate run troubleshoot

Jupyter command `jupyter-nbconvert` not found.
